# SAGE on DisjunctiveNet: Kaggle Benchmark Runner
Notebook này chạy gọn nhẹ bằng cách import trực tiếp package `disnet_sage` từ file zip hoặc GitHub repo.


In [ ]:
# 1. SETUP MÃ NGUỒN TRÊN KAGGLE
import os, sys

# Cách 1: Giải nén file zip nếu bạn upload file disnet_sage_kaggle.zip lên Kaggle
if os.path.exists("disnet_sage_kaggle.zip"):
    !unzip -q -o disnet_sage_kaggle.zip -d .
    print("✅ Đã giải nén disnet_sage_kaggle.zip")
elif os.path.exists("/kaggle/input"):
    # Tìm file zip trong Kaggle datasets nếu có
    import glob
    zips = glob.glob("/kaggle/input/**/disnet_sage_kaggle.zip", recursive=True)
    if zips:
        !unzip -q -o {zips[0]} -d .
        print(f"✅ Đã giải nén {zips[0]}")
    elif not os.path.exists("disnet_sage"):
        !git clone https://github.com/ghugn/SAGE_ICML2027.git
        %cd SAGE_ICML2027
elif not os.path.exists("disnet_sage"):
    !git clone https://github.com/ghugn/SAGE_ICML2027.git
    %cd SAGE_ICML2027

sys.path.insert(0, os.getcwd())
print("Thư mục hiện tại:", os.getcwd())


In [ ]:
# 2. CHẠY SMOKETEST NHANH (5 EPOCHS)
!python smoketest.py --epochs 5


In [ ]:
# 3. CHẠY THỰC NGHIỆM ĐỐI SÁNH ĐẦY ĐỦ DISNET VS SAGE (100 EPOCHS)
!python demo_disnet_sage.py


In [ ]:
# 4. HOẶC IMPORT VÀ SỬ DỤNG TRỰC TIẾP TRONG NOTEBOOK
import torch
import torch.nn as nn
from disnet_sage import DisjunctiveModel, LinearConstraint, constrained_model

# Khởi tạo mô hình
dm = DisjunctiveModel(2)
dm.set_bounds(lower=[0.0, 0.0], upper=[1.0, 1.0])
dm.add_linear_constraint([1.0, 1.0], ">=", 0.8)
dm.add_disjunction(
    [LinearConstraint([1.0, 0.0], "<=", 0.2)],
    [LinearConstraint([1.0, 0.0], ">=", 0.4), LinearConstraint([1.0, 0.0], "<=", 0.6)],
    [LinearConstraint([1.0, 0.0], ">=", 0.8)],
)
dm.add_disjunction(
    [LinearConstraint([0.0, 1.0], "<=", 0.2)],
    [LinearConstraint([0.0, 1.0], ">=", 0.35), LinearConstraint([0.0, 1.0], "<=", 0.55)],
    [LinearConstraint([0.0, 1.0], ">=", 0.7)],
)

backbone = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
model = constrained_model(backbone, dm, layer_type="sage", rho=0.25, n_samples=32)

x = torch.tensor([0.2, 0.7, 0.4])
with torch.no_grad():
    pred = model(x)
print("Prediction:", pred.numpy().round(4), "| Feasible (sum>=0.8):", bool(pred.sum().item() >= 0.799))
